# core

> Fill in a module description here

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

## Database

In [ ]:
from fastcore.all import *

In [ ]:
class Video: id:int; title:str=''; overview:str=''; transcript:str=''; length:int=0; sample_rate:int=0; path:str=''
class Frame: id:int; video_id:int; frame_number:int; subtitle:str=''
class Run: id:int; deploy_time:str; finish_time:str; request_timer:str; video_id:int; model:str; usage:str; num_frames:int; description:str
class RunFrame: run_id:int; frame_id:int; type:str; system_prompt:str; prompt:str; description:str; usage:str

In [ ]:
from fastlite import *

In [ ]:
!rm db.db
db = database('db.db'); db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
??Queryable.schema

Type:        property
String form: <property object>
Source:     
# Queryable.schema.fget
@property
def schema(self) -> str:
    "SQL schema for this table or view."
    return self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0]

In [ ]:
@patch(as_prop=True)
def schema(self:Queryable) -> str:
    "SQL schema for this table or view."
    return hl_md(self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0], lang='sql')

In [ ]:
videos = db.create(Video, transform=True); videos.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

In [ ]:
frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')]); frames.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [frame] (
   [id] INTEGER PRIMARY KEY,
   [video_id] INTEGER REFERENCES [video]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_number] INTEGER,
   [subtitle] TEXT
)
```

</div>

In [ ]:
runs = db.create(Run, transform=True); runs.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run] (
   [id] INTEGER PRIMARY KEY,
   [deploy_time] TEXT,
   [finish_time] TEXT,
   [request_timer] TEXT,
   [video_id] INTEGER,
   [model] TEXT,
   [usage] TEXT,
   [num_frames] INTEGER,
   [description] TEXT
)
```

</div>

In [ ]:
runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True); runframes.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run_frame] (
   [run_id] INTEGER REFERENCES [run]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_id] INTEGER REFERENCES [frame]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [type] TEXT,
   [system_prompt] TEXT,
   [prompt] TEXT,
   [description] TEXT,
   [usage] TEXT,
   PRIMARY KEY ([run_id], [frame_id], [type])
)
```

</div>

In [ ]:
from apswutils.db import Database

In [ ]:
def init_db(
    path:str|Path='db.db' # Path to database
) -> Database:
    "Initialize a database and return it."
    db = database(path)
    db.create(Video, transform=True)
    db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')])
    db.create(Run, transform=True)
    db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True)
    return db
    

In [ ]:
# !rm db.db
db = init_db()

In [ ]:
db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
db.t.video.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

## VLM

In [ ]:
from fastllm.types import Msg, Text, InputImage, Thinking

In [ ]:
?Msg

````python
def Msg(
    role:str, content:List
)->None:
    "A normalized message."
````

**File:** `~/.local/lib/python3.12/site-packages/aidialog/msg_parts.py`; line: 93

**Type:** type

In [ ]:
?Text

````python
def Text(
    text:str=None, citations:list=None, raw:dict=None, cache_control:dict=None
)->None:
    "Plain text content."
````

**File:** `~/.local/lib/python3.12/site-packages/aidialog/msg_parts.py`; line: 39

**Type:** type

In [ ]:
?InputImage

````python
def InputImage(
    text:str=None, mime:str=None, raw:dict=None, cache_control:dict=None
)->None:
    "An image input."
````

**File:** `~/.local/lib/python3.12/site-packages/aidialog/msg_parts.py`; line: 62

**Type:** type

In [ ]:
def user(
    txt:str,
    img:str|None=None,
)->Msg:
    "Build a user message with optional image."
    if img is None: return Msg(role='user', content=[Text(txt)])
    else: return Msg(role='user', content=[InputImage(img), Text(txt)])

In [ ]:
user('你好')

**Msg**

- role: `user`

<contents>

**Text** (`text`)

你好

<details markdown='1'>

- raw: `None`
- citations: `None`

</details>

</contents>

In [ ]:
def assistant(
    txt:str,
    citations:list=None,
)->Msg:
    "Build an assistant message."
    return Msg(role='assistant', content=[Text(txt, citations=citations)])

In [ ]:
assistant('嗨')

**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

嗨

<details markdown='1'>

- raw: `None`
- citations: `None`

</details>

</contents>

In [ ]:
from fastllm.acomplete import acomplete

In [ ]:
?acomplete

````python
async def acomplete(
    msgs, model, api_name:NoneType=None, vendor_name:NoneType=None, api_key:NoneType=None, base_url:NoneType=None,
    xtra_body:NoneType=None, xtra_hdrs:NoneType=None, stream:bool=False, stop_callables:NoneType=None, retries:int=2,
    retry_delay:float=0.5, system:NoneType=None, max_tokens:NoneType=None, temperature:NoneType=None,
    tools:NoneType=None, tool_choice:NoneType=None, reasoning_effort:NoneType=None, web_search_options:NoneType=None
):
    "Unified completion across different APIs."
````

**File:** `~/.local/lib/python3.12/site-packages/fastllm/acomplete.py`; line: 170

**Type:** function

In [ ]:
from cachy import enable_cachy, disable_cachy

In [ ]:
enable_cachy()
await acomplete([user('hi')], 'deepseek-v4-flash', vendor_name='deepseek')

<details><summary>Thinking</summary>

好的，用户只发了一个“hi”，这是非常简单的打招呼。用户可能刚进入对话，想测试我是否在线或者开始一个友好的交流。深层需求应该是希望得到热情、友好的回应，开启一次对话。我不需要复杂分析，直接礼貌问候并表达乐于助人的态度，用开放式的邀请让用户提出具体问题。想到了用“你好！”开头，加上表情符号显得亲切，然后自我介绍并说明能力范围，最后用提问引导对话继续。

</details>

你好！很高兴见到你！😊

有什么我可以帮你的吗？无论是回答问题、帮你整理信息、提供创作灵感，还是聊聊天，我都很乐意陪你一起。你只需告诉我需求，剩下的交给我！

<details markdown='1'>

- model: `deepseek-v4-flash`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=5, completion_tokens=143, total_tokens=148, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=97, raw={'prompt_tokens': 5, 'completion_tokens': 143, 'total_tokens': 148, 'prompt_tokens_details': {'cached_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 97}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 5})`

</details>

In [ ]:
from fastllm.types import Completion
async def stream(
    msgs:list|None=None,  # Messages to send
    model:str='',  # Model name (e.g. 'deepseek-v4-flash')
    max_think:float=float('inf'),  # Max thinking tokens to display
    usage:bool=True,  # Show usage info in output
    display:bool=True,  # Print text/thinking as it arrives
    **kwargs,  # Passed to acomplete
) -> Completion:  # Return the final completion
    "Stream a response, printing text/thinking as it arrives. Returns the final completion."
    assert msgs is not None, 'no messages provided'
    assert model!='', 'no model name provided'
    think_cnt, seen_txt = 0, False
    async for o in await acomplete(msgs, model, stream=True, **kwargs):
        if not isinstance(o, Completion) and display:
            if isinstance(o, Thinking) and think_cnt<max_think: print('🤔', end='', flush=True)
            if isinstance(o, Text) and (txt:=o.text): print(f"{'\n\n' if not seen_txt else ''}{txt}", end='', flush=True) or not seen_txt and (seen_txt:=True)
            think_cnt+=1
    if display: print()
    return o

In [ ]:
r = await stream([user('hi')], 'deepseek-v4-flash', vendor_name='deepseek')

In [ ]:
from base64 import b64encode
def img2b64(
    path:Path,
)->str:
    "Encode an image file as a base64 data URL."
    return 'data:image/png;base64,'+b64encode(Path.read_bytes(path)).decode()

In [ ]:
r = await stream([user('what do ye elf eyes see', img2b64(Path('./test.jpg')))], 'bytedance-seed/seed-2.0-lite', vendor_name='openrouter', reasoning_effort='high')



*

til

ts

 point

y

...

 our

 wild

 green

 space

.

In [ ]:
@delegates(stream, keep=True)
def session(
    **kwargs,
):
    "Create a stream partial with preset model/kwargs."
    return partial(stream, **kwargs)

## Run

In [ ]:
#| export
async def _process_frame(db, session, run_id:int, video_id:int, p:Path, prompt:str, prompt_type:str):
    "Run VLM on a single frame and store the result."
    r = await session([user(prompt, img=img2b64(p))])
    fnum = int(p.stem.split('_')[1])
    frame = db.t.frame.selectone('video_id=? AND frame_number=?', (video_id, fnum))
    db.t.run_frame.insert(run_id=run_id, frame_id=frame.id, type=prompt_type, prompt=prompt, description=r.message.text, usage=r.usage.raw)

In [ ]:
#| export
from datetime import datetime
from zoneinfo import ZoneInfo

tz = ZoneInfo('Asia/Hong_Kong')

def _run_header(run, model:str, start:int, stop:int, step:int, cache:bool):
    "Print run header box."
    print(f'╭─ Run #{run.id} ═══════════════════════════════╮\n│ Model    {model}\n│ Start    {start}\n│ Stop     {stop}\n│ Step     {step}\n│ Frames   {(stop-start)//step}\n│ Cache    {cache}\n│ Time     {datetime.now(tz).strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')

In [ ]:
#| export
def compute_usage(db, run_id:int)->str:
    "Aggregate usage stats across all frames in a run. Returns JSON string."
    usgs = L(db.t.run_frame.rows_where('run_id=?', (run_id,))).map(lambda r: loads(r['usage']))
    if not usgs: return '{}'
    tot = {k: ({k2:0 for k2 in v} if isinstance(v,dict) else 0) for k,v in usgs[0].items()}
    for u in usgs:
        for k,v in u.items():
            if isinstance(v,dict):
                for k2,v2 in v.items(): tot[k][k2] += v2
            else: tot[k] += v
    return dumps(tot)

def _finish_run(db, run_id:int)->Run:
    "Update run with finish time and usage, print summary box."
    db.t.run.update(id=run_id, finish_time=datetime.now(tz), request_timer=None, usage=compute_usage(db, run_id))
    run = db.t.run[run_id]
    tot = dict2obj(loads(run.usage))
    elapsed = datetime.fromisoformat(run.finish_time) - datetime.fromisoformat(run.deploy_time)
    print(f'╭─ Run #{run.id} Complete ═════════════════════╮\n│ Finish   {datetime.fromisoformat(run.finish_time).strftime("%H:%M:%S")}\n│ Elapsed  {str(elapsed).split(".")[0]}\n│ Cost     ${tot.cost:.4f} (HKD {tot.cost*7.84:.2f})\n╰──────────────────────────────────────────────╯')
    return run

In [ ]:
#| export
def _run_header(run, model:str, start:int, stop:int, step:int, cache:bool):
    "Print run header box."
    print(f'╭─ Run #{run.id} ═══════════════════════════════╮\n│ Model    {model}\n│ Start    {start}\n│ Stop     {stop}\n│ Step     {step}\n│ Frames   {(stop-start)//step}\n│ Cache    {cache}\n│ Time     {datetime.now(tz).strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')

In [ ]:
#| export
from typing import Callable
from fastprogress.fastprogress import NBMasterBar as master_bar
async def deploy_run(
    video_id:int,
    db:Database,
    session:Callable,
    prompt:str,
    prompt_type:str,
    start:int=0,
    stop:int|None=None,
    step:int=1,
    cache:bool=False,
)->Run:
    "Run a single prompt across a range of frames, storing results in the database."
    video = db.t.video[video_id]
    fpath = L(Path(video.path).glob('frame_*.jpg')).sorted(key=~Self.stem.split('_'))
    if stop is None: stop = len(fpath)
    if not cache: disable_cachy(); print('!! Cache disabled')
    else: print('!! Using cache')

    run = db.t.run.insert(deploy_time=datetime.now(tz), video_id=video_id, model=session.keywords['model'], num_frames=(stop-start)//step)
    _run_header(run, session.keywords['model'], start, stop, step, cache)

    for p in (mb:=master_bar(fpath[start:stop:step])):
        mb.main_bar.comment = f'frame {p.stem.split("_")[1]}'
        await _process_frame(db, session, run.id, video_id, p, prompt, prompt_type)

    if not cache: enable_cachy(); print('!! Cache enabled')
    return _finish_run(db, run.id)

## Summary

In [ ]:
#| export
from tiktoken import encoding_for_model

In [ ]:
?encoding_for_model

````python
def encoding_for_model(
    model_name:str
)->Encoding:

````

````
Returns the encoding used by a model.

Raises a KeyError if the model name is not recognised.
````

**File:** `/usr/local/lib/python3.12/site-packages/tiktoken/model.py`; line: 113

**Type:** function

In [ ]:
#| export
def _build_window(db, run_id:int, start:int, stop:int, step:int=1)->tuple[str,int]:
    "Query runframes and build the window text with token count."
    rows = L(db.t.run_frame.rows_where(where='run_id=? AND frame_id>=? AND frame_id<?', where_args=(run_id, start, stop)))
    grouped = rows.groupby(lambda r: r['frame_id'])
    fids = sorted(grouped.keys())
    if step > 1: fids = fids[::step]

    window = ''
    for fid in fids:
        prefix = f'\n\nTimestamp ({fid}s)\n'
        window += prefix + len(prefix.strip())*'='
        for d in grouped[fid]: window += f"\n\n--\n\n{d['type'].upper()}\n\n{d['description']}"

    enc = encoding_for_model('gpt-4o')
    return window, len(enc.encode(window))

In [ ]:
#| export
def _summary_header(run_id:int, start:int, stop:int, step:int, model:str, window:str, win_tokens:int, cache:bool, t0:datetime):
    "Print summary run header box."
    print(f'╭─ Summary Run #{run_id} ═══════════════════════╮\n│ Frames   {start}–{stop} (step {step})\n│ Model    {model}\n│ Window   {len(window)} chars / {win_tokens} tokens\n│ Cache    {cache}\n│ Start    {t0.strftime("%H:%M:%S")}\n╰──────────────────────────────────────────────╯')

In [ ]:
#| export
def _summary_footer(summary:str, win_tokens:int, t0:datetime, t1:datetime, cost:float):
    "Print summary completion box."
    enc = encoding_for_model('gpt-4o')
    sum_tokens = len(enc.encode(summary))
    reduction = (1 - sum_tokens/win_tokens)*100 if win_tokens else 0
    elapsed = t1 - t0
    print(f'╭─ Summary Complete ═══════════════════════════╮\n│ Finish   {t1.strftime("%H:%M:%S")}\n│ Elapsed  {str(elapsed).split(".")[0]}\n│ Summary  {len(summary)} chars / {sum_tokens} tokens\n│ Reduced  {reduction:.1f}%\n│ Cost     ${cost:.4f} (HKD {cost*7.84:.2f})\n╰──────────────────────────────────────────────╯')

In [ ]:
#| export
async def summarize_window(
    db:Database,
    run_id:int,
    start:int,
    stop:int,
    session,
    sys_prompt:str,
    step:int=1,
    cache:bool=False,
)->str:
    "Summarize a window of frames from runframes. Returns summary text."
    if not cache: disable_cachy()

    window, win_tokens = _build_window(db, run_id, start, stop, step)
    t0 = datetime.now(tz)
    _summary_header(run_id, start, stop, step, session.keywords['model'], window, win_tokens, cache, t0)

    r = await session([user(window)], system=sys_prompt)
    t1 = datetime.now(tz)

    if not cache: enable_cachy()

    summary = r.message.text
    _summary_footer(summary, win_tokens, t0, t1, r.usage.raw.get('cost', 0))
    return summary

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()